In [1]:
import os,time,gc,re
import pandas as pd
import numpy as np
import shutil
import xgboost as xgb
from zipfile import ZipFile
import subprocess
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score,average_precision_score
from google.colab import drive
from pathlib import Path




In [ ]:
!nvidia-smi

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
Path('/content/drive/MyDrive/HomeCredit_XGBoost/Data').mkdir(parents=True,exist_ok=True)
os.makedirs('/content/drive/MyDrive/HomeCredit_XGBoost/outputs',exist_ok=True)

In [4]:
DATA_DIR = '/content/data'
OUT_DIR='/content/drive/MyDrive/HomeCredit_XGBoost/outputs'
SEED=42
N_FOLDS=5

def detect_device():
    if shutil.which('nvidia-smi') is None:
        return 'cpu'
    r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
    return 'cuda' if r.returncode == 0 and 'GPU 0' in r.stdout else 'cpu'

device = detect_device()


# device='cuda' if shutil.which('nvidia-smi') else 'cpu'

DEBUG=False
DEBUG_ROWS=20_000
log_wandb=True
WANDB_PROJECT = 'home-credit-default-risk'

os.makedirs(OUT_DIR,exist_ok=True)
print(f'XGBOOST {xgb.__version__} | pandas version {pd.__version__} | device {device}')



XGBOOST 3.4.1 | pandas version 2.2.3 | device cuda


In [ ]:
import os
try:
    from google.colab import userdata
    # os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:

import requests

COMP = 'home-credit-default-risk'

# /content/data on Colab, local project dir otherwise
IN_COLAB = os.path.exists('/content')
DATA_DIR = '/content/data' if IN_COLAB else os.path.join('data', 'home-credit')

EXPECTED_FILES = [
    'application_train.csv',
    'application_test.csv',
    'bureau.csv',
    'bureau_balance.csv',
    'credit_card_balance.csv',
    'installments_payments.csv',
    'POS_CASH_balance.csv',
    'previous_application.csv',
    'sample_submission.csv',
]


def download_competition(dest_zip):
    """Fetch the competition archive using a Kaggle bearer token."""
    token = os.environ.get('KAGGLE_KEY', '').strip()
    if not token:
        raise RuntimeError('KAGGLE_KEY not set — add it as a Colab secret or env var')

    url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMP}'
    headers = {'Authorization': f'Bearer {token}'}

    with requests.get(url, headers=headers, stream=True, timeout=120) as resp:
        if resp.status_code == 403:
            raise RuntimeError(f'403 — accept the rules at kaggle.com/competitions/{COMP}/rules')
        resp.raise_for_status()

        total = int(resp.headers.get('Content-Length') or 0)
        done = 0
        with open(dest_zip, 'wb') as fh:
            for chunk in resp.iter_content(1 << 20):
                fh.write(chunk)
                done += len(chunk)
                pct = f'{done / 1e6:.0f}/{total / 1e6:.0f} MB' if total else f'{done / 1e6:.0f} MB'
                print(f'\r{pct}', end='')
        print()


os.makedirs(DATA_DIR, exist_ok=True)
zip_path = os.path.join(DATA_DIR, f'{COMP}.zip')

missing = [f for f in EXPECTED_FILES if not os.path.exists(os.path.join(DATA_DIR, f))]

if not missing:
    print(f'All {len(EXPECTED_FILES)} CSVs found in {DATA_DIR} — skipping download')
else:
    print(f'Missing {len(missing)} file(s): {missing}')

    if os.path.exists(zip_path):
        print(f'Archive already present: {zip_path} — skipping download')
    else:
        print('Downloading from Kaggle...')
        download_competition(zip_path)

    print('Extracting...')
    with ZipFile(zip_path) as archive:
        archive.extractall(DATA_DIR)
    print('Extraction complete')


**HelperMethods**

In [ ]:
print(DATA_DIR)

In [5]:
@contextmanager
def timer(name):
    """Print how long a block took"""
    t0=time.time()
    #try finally to prevent the propogation of exception past the yield
    try:
        yield
    finally:
            print(f'[{name}] {time.time()-t0:.0f}s')

def read(name):
     """Reads one csv, nrows equals debug rows if debug true"""
     nrows=DEBUG_ROWS if DEBUG else None
     return pd.read_csv(os.path.join(DATA_DIR,f'{name}.csv'),nrows=nrows)

def is_text(s):
     """True for string columns: object dtype (pandas 2) or 'str' dtype (pandas 3)."""
     return s.dtype == object or isinstance(s.dtype,pd.StringDtype)

def reduce_mem(df):
    """float64 → float32 and int64 → smallest int."""
    for col in df.columns:
        if col.startswith('SK_ID'):
            continue   # pandas 2.3.2 cannot factorize int32 join keys, also this is used as join key and join key should be stable not inferred
        dt=df[col].dtype
        if dt == np.float64:
            df[col]=df[col].astype(np.float32)
        elif dt== np.int64:
            df[col]=pd.to_numeric(df[col],downcast='integer')
    return df

def safe_div(a,b):
    """a / b, with NaN instead of inf when b == 0. pandas mean skips nan but propogates inf. One inf inside a client's
    rows would otherwise turn that client's mean into inf."""

    return a/b.where( b!= 0)

def one_hot(df):
    """One-hot encode every text column of a CHILD table. return df , and  the new columns added"""
    original=list(df.columns)
    cat_cols=[c for c in original if is_text(df[c])]
    df=pd.get_dummies(df,columns=cat_cols,dummy_na=True,dtype=np.uint8)
    new_cols=[c for c in df.columns if c not in original]
    return df, new_cols
# note , not on the main table . child table must be
# aggregated to one row per client, and the mean of a 0/1 column is a
# fraction. A raw text    column cannot be averaged. dummy_na=True adds a column for missing values. also get dummies is used becuase none of these values will be forwarded to the model , so we can have stateless encoding
# this exist for the purpose so that aggregate cna take their mean and turn them into fractions. also there is no train /test boundary at this point. also get dummies take the dataframe and returns dataframe. whereas the one hot encoder returns numpy array
#we would need get feature names out and manual rebuild , we cant groupy a sparse matrix, we would need column transformer  to keep the key attatched


def aggregate(df,key,spec,prefix):
    """groupby(key).agg(spec) with flat, readable column names.
         spec = {column: [statistics]}, → columns PREFIX_DPD_MAX, PREFIX_DPD_MEAN, indexed by `key`.
         Columns missing from df are skipped (matters only in DEBUG mode)."""
    spec= {c:s for c,s in spec.items() if c in df.columns}
    out=df.groupby(key).agg(spec) #return multinindex columns, key becomes the index
    out.columns=[ f'{prefix}_{col}_{stat}'.upper() for col,stat in out.columns]
    return out





In [ ]:
# # ---------------------------------------------------------------------------
# # Data scan — find DISGUISED missing values.TODO
# #
# # Values like 'XNA' or 365243 mean "missing" but are invisible to isna():
# # application_train.CODE_GENDER.isna().sum() is 0, yet 4 rows hold 'XNA', and
# # nothing in HomeCredit_columns_description.csv documents either one.
# #
# # Method: for every column print BOTH ends of its frequency distribution, and
# # for numeric columns both ends of its RANGE as well.
# #   most common  -> catches sentinels that dominate (DAYS_EMPLOYED 365243, 18%)
# #   least common -> catches sentinels that are rare  (CODE_GENDER 'XNA', 4 rows)
# #   range + repeat count -> catches the ones that are neither (see `extremes`)
# # Nothing is hardcoded: no assumed list of placeholder tokens, so every finding
# # comes from reading the data rather than from a guess about this vendor.
# #
# # Judgement is still yours. 'Maternity leave' x5 is a real category; 'XNA' x4
# # is a placeholder. `receipt` below is how you tell the two apart with evidence.
# # ---------------------------------------------------------------------------
# RUN_SCAN   = True     # set False once you have read the output
# MAX_LEVELS = 25       # above this a column is high-cardinality: its rarest
#                       # value is just some float seen once, so report the count

# SCAN_TABLES = ['application_train','application_test','bureau','bureau_balance',
#                'previous_application','POS_CASH_balance','installments_payments',
#                'credit_card_balance']


# def fmt(v):
#     """repr() for strings so quotes and stray whitespace stay visible; plain
#     for everything else (numpy 2 renders repr(np.int32(1)) as 'np.int32(1)',
#     which is pure noise here)."""
#     return repr(v) if isinstance(v, str) else v


# def extremes(s):
#     """For a high-cardinality NUMERIC column: its range, and how often each end
#     REPEATS. This is the gap the mode/rarest pair cannot cover. A sentinel
#     sitting at 2% is neither the most nor the least common value, so it hides in
#     the middle of the frequency distribution — but it is always at one end of
#     the RANGE. Worked case: an AMT_ column that is 60% zeros reports mode=0.0
#     and rarest=<some float> x1, while 200 rows of 99999 go unmentioned.
#     The repeat count is the tell. A genuine extreme occurs once or twice; a
#     placeholder occurs thousands of times, because it is a code, not a
#     measurement. Then read the range against what the column is documented to
#     mean — a DAYS_ column (days BEFORE the application) with a positive max is
#     an encoding, whatever the number turns out to be."""
#     v = s.dropna()
#     if v.empty:
#         return ''
#     lo, hi = v.min(), v.max()
#     return (f'min={lo:<12g} x{int((v == lo).sum()):<8,} '
#             f'max={hi:<12g} x{int((v == hi).sum()):,}')


# def scan(tables=SCAN_TABLES):
#     """Both ends of every column's distribution, for every table.
#     dropna=False matters: without it OCCUPATION_TYPE reports 'Laborers' as its
#     mode and hides that 31% of the column is actually NaN.
#     Read raw, with no reduce_mem: casting float64 to float32 merges values that
#     differ only in the low bits, which changes the very counts being inspected.
#     Profile the data as it arrived, not as you reshaped it."""
#     if DEBUG:
#         print(f'WARNING: DEBUG is True, read() truncates to {DEBUG_ROWS:,} rows '
#               '— 4-row findings like CODE_GENDER XNA will be missed\n')
#     for name in tables:
#         df = read(name)
#         print(f'=== {name}  ({len(df):,} rows x {df.shape[1]} cols) ===')
#         for c in df.columns:
#             vc = df[c].value_counts(dropna=False)
#             if len(vc) <= MAX_LEVELS:
#                 tail = f'rarest={fmt(vc.index[-1])} x{vc.iloc[-1]:,}'
#             elif is_text(df[c]):
#                 tail = f'{len(vc):,} distinct'
#             else:
#                 tail = f'{len(vc):,} distinct   {extremes(df[c])}'
#             print(f'  {c:28} mode={str(fmt(vc.index[0]))[:22]:22} '
#                   f'{vc.iloc[0]/len(df):6.1%}   {tail}')
#         print()
#         del df
#         gc.collect()


# def receipt(table, col, against):
#     """Prove what a suspicious value MEANS, instead of asserting it.
#     receipt('previous_application','CODE_REJECT_REASON','NAME_CONTRACT_STATUS')
#     shows XAP only ever occurs on Approved/Canceled rows — so it is 'not
#     applicable' and INFORMATIVE — while XNA sits almost entirely on Refused
#     rows, so it is 'not available' and genuinely missing. Same tool proved
#     DAYS_EMPLOYED==365243 and ORGANIZATION_TYPE=='XNA' mark identical rows.
#     Real NaN is kept as its own row and column rather than dropped, because the
#     question being asked is precisely whether the suspicious value behaves like
#     NaN or unlike it — and pd.crosstab discards NaN by default."""
#     df = read(table)
#     show = lambda s: s.astype(object).where(s.notna(), '(NaN)')
#     print(pd.crosstab(show(df[col]), show(df[against])).to_string())
#     del df
#     gc.collect()


# if RUN_SCAN:
#     with timer('scan'):
#         scan(SCAN_TABLES)        # widen to SCAN_TABLES for all 7

**Main Table**
column names for columns description file

In [ ]:
def application_features():
      train=read('application_train')
      test=read('application_test')
      print(f"train {train.shape} | test {test.shape} | "
            f"positive rate {train['TARGET'].mean():.4f}")

      df=pd.concat([train,test],ignore_index=True) #no leakage
      del train,test
      gc.collect()

      #cleaning

      #gender
      df['CODE_GENDER']=df["CODE_GENDER"].mask(df['CODE_GENDER']=='XNA')

      anom=df["DAYS_EMPLOYED"] == 365243
      df["DAYS_EMPLOYED_ANOM"] = anom.astype(np.uint8) #having no value is a signal , so adding this. order matters. flag , keep the flag and then impute
      df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].mask(anom)

      #ratios
      df["CREDIT_INCOME_RATIO"] = safe_div(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL"])
      df["ANNUITY_INCOME_RATIO"] = safe_div(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL"])
      # per-period payment / loan amount ≈ 1 / (number of payments):
      # a proxy for loan term, which the table does not give directly
      df["PAYMENT_RATE"] = safe_div(df["AMT_ANNUITY"], df["AMT_CREDIT"])
      # < 1 → loan is bigger than the goods price (extra borrowed on top)
      df["GOODS_CREDIT_RATIO"] = safe_div(df["AMT_GOODS_PRICE"], df["AMT_CREDIT"])
      # fraction of life spent at the current job (both are negative days)
      df["EMPLOYED_AGE_RATIO"] = safe_div(df["DAYS_EMPLOYED"], df["DAYS_BIRTH"])
      df["INCOME_PER_PERSON"] = safe_div(df["AMT_INCOME_TOTAL"], df["CNT_FAM_MEMBERS"])
      #normalized credit scores from outside agencies, per client summaries
      ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
      df["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
      df["EXT_SOURCE_STD"] = ext.std(axis=1)
      df["EXT_SOURCE_MIN"] = ext.min(axis=1)
      df["EXT_SOURCE_MAX"] = ext.max(axis=1)
      df["EXT_SOURCE_PROD"] = ext.prod(axis=1, min_count=3)  # NaN unless all 3 exis

      return reduce_mem(df)





In [ ]:
def bureau_features():
    bureau = reduce_mem(read("bureau"))
    bb = reduce_mem(read("bureau_balance"))

    #  months → loan
    bb,bb_cat=one_hot(bb)
    #min -> how far back the record goes,
    # max -> how recently it was updated
    # size -> months of observed history.
    bb_spec = {"MONTHS_BALANCE": ["min", "max", "size"]}
    # severe = [c for c in bb_cat if c.endswith(("_3", "_4", "_5"))] #Months balance is untouched so fine now, but if any new cols are added this may break
    severe = [c for c in bb_cat if c in ("STATUS_3", "STATUS_4", "STATUS_5")] #explicitly on status,
    bb_spec.update({c: ["mean"] for c in bb_cat})
    bb_spec.update({c: ["mean", "max"] for c in severe}) #making defualt explicity , max means if this loadn was ever 120+ days overdue or written off
    bb_agg = aggregate(bb, "SK_ID_BUREAU", bb_spec, "BB")
    del bb
    gc.collect()

    bureau = bureau.join(bb_agg, on="SK_ID_BUREAU", how="left") #left keeps the loans not in bb as well
    bureau = bureau.drop(columns="SK_ID_BUREAU") #not needed now
    del bb_agg
    gc.collect()

    # loans -> client
    bureau, bureau_cat = one_hot(bureau)
    num_spec = {
        "DAYS_CREDIT": ["min", "max", "mean", "count"],  # count = number of loans
        "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
        "DAYS_CREDIT_UPDATE": ["mean"],
        "CREDIT_DAY_OVERDUE": ["max", "mean"],
        "AMT_CREDIT_MAX_OVERDUE": ["mean"],
        "AMT_CREDIT_SUM": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_DEBT": ["max", "mean", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["mean"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "sum"],
        "AMT_ANNUITY": ["max", "mean"],
        "CNT_CREDIT_PROLONG": ["sum"],
        "BB_MONTHS_BALANCE_MIN": ["min"],
        "BB_MONTHS_BALANCE_MAX": ["max"],
        "BB_MONTHS_BALANCE_SIZE": ["mean", "sum"],
    }
    bb_status_cols = [c for c in bureau.columns if c.startswith("BB_STATUS")] #already one hotted
    cat_spec = {c: ["mean"] for c in bureau_cat + bb_status_cols}
    out = aggregate(bureau, "SK_ID_CURR", {**num_spec, **cat_spec}, "BURO")

    if "CREDIT_ACTIVE_Active" in bureau: #getting current burden , max debt accross all bureau loans includes closed as well
        active = bureau[bureau["CREDIT_ACTIVE_Active"] == 1]
        out = out.join(aggregate(active, "SK_ID_CURR", num_spec, "BURO_ACTIVE"))
    if "CREDIT_ACTIVE_Closed" in bureau:#measures track record
        closed = bureau[bureau["CREDIT_ACTIVE_Closed"] == 1]
        out = out.join(aggregate(closed, "SK_ID_CURR", num_spec, "BURO_CLOSED"))

    # Ratio after aggregation: share of all bureau credit still owed.
    out["BURO_DEBT_CREDIT_RATIO"] = safe_div(
        out["BURO_AMT_CREDIT_SUM_DEBT_SUM"], out["BURO_AMT_CREDIT_SUM_SUM"])

    del bureau
    gc.collect()
    return out


In [ ]:
def previous_features():
    prev = reduce_mem(read("previous_application"))

    # Same 365243 placeholder as in the main table.
    for c in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
              "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
        prev[c] = prev[c].mask(prev[c] == 365243)

    # asked / granted: > 1 means the client got less than they asked for —
    # the lender already judged them riskier than they judged themselves.
    prev["APP_CREDIT_RATIO"] = safe_div(prev["AMT_APPLICATION"], prev["AMT_CREDIT"])

    prev, prev_cat = one_hot(prev)

    num_spec = {
        "AMT_ANNUITY": ["min", "max", "mean"],
        "AMT_APPLICATION": ["min", "max", "mean"],
        "AMT_CREDIT": ["min", "max", "mean"],
        "APP_CREDIT_RATIO": ["min", "max", "mean", "var"], #var captures inconsitency
        "AMT_DOWN_PAYMENT": ["min", "max", "mean"],
        "AMT_GOODS_PRICE": ["min", "max", "mean"],
        "HOUR_APPR_PROCESS_START": ["min", "max", "mean"],
        "RATE_DOWN_PAYMENT": ["min", "max", "mean"],
        "DAYS_DECISION": ["min", "max", "mean", "count"],  # count = n applications
        "CNT_PAYMENT": ["mean", "sum"],
    }
    # Fractions like PREV_NAME_CONTRACT_STATUS_REFUSED_MEAN come out of this.
    cat_spec = {c: ["mean"] for c in prev_cat}

    out = aggregate(prev, "SK_ID_CURR", {**num_spec, **cat_spec}, "PREV")

    # Split by outcome: what the client asked for when APPROVED vs REFUSED
    # tells different stories (e.g. repeated large refused requests).
    if "NAME_CONTRACT_STATUS_Approved" in prev:
        approved = prev[prev["NAME_CONTRACT_STATUS_Approved"] == 1]
        out = out.join(aggregate(approved, "SK_ID_CURR", num_spec, "PREV_APPROVED"))
    if "NAME_CONTRACT_STATUS_Refused" in prev:
        refused = prev[prev["NAME_CONTRACT_STATUS_Refused"] == 1]
        out = out.join(aggregate(refused, "SK_ID_CURR", num_spec, "PREV_REFUSED"))

    del prev
    gc.collect()
    return out


In [ ]:
def pos_cash_features():
    pos = reduce_mem(read("POS_CASH_balance"))
    pos, pos_cat = one_hot(pos)

    spec = {
        "MONTHS_BALANCE": ["max", "mean", "size"],
        # SK_DPD = days past due that month; SK_DPD_DEF = same, but ignoring
        # tiny debts (the "real" delinquency). max = worst month ever.
        "SK_DPD": ["max", "mean"],
        "SK_DPD_DEF": ["max", "mean"],
        "CNT_INSTALMENT_FUTURE": ["mean"],  # instalments still left to pay
    }
    spec.update({c: ["mean"] for c in pos_cat})

    out = aggregate(pos, "SK_ID_CURR", spec, "POS")
    out["POS_N_LOANS"] = pos.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique() #distinct rows ,size counts monthly rows

    del pos
    gc.collect()
    return out

In [ ]:
def installments_features():
    ins = reduce_mem(read("installments_payments"))

    # Amount behaviour
    ins["PAYMENT_RATIO"] = safe_div(ins["AMT_PAYMENT"], ins["AMT_INSTALMENT"])  # < 1 -> underpaid
    ins["PAYMENT_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]           # > 0 -> shortfall

    # Timing behaviour. Both columns are negative days, so
    # entry - due > 0 means the payment came AFTER the due date.
    #   DPD = days past due (late), DBD = days before due (early), both >= 0.
    ins["DPD"] = (ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]).clip(lower=0) #zero if eaely
    ins["DBD"] = (ins["DAYS_INSTALMENT"] - ins["DAYS_ENTRY_PAYMENT"]).clip(lower=0) #zero if late
    ins["LATE"] = (ins["DPD"] > 0).astype(np.uint8)  # mean → fraction paid late

    spec = {
        "NUM_INSTALMENT_VERSION": ["nunique"],  # how often the schedule changed
        "DPD": ["max", "mean", "sum"],
        "DBD": ["max", "mean", "sum"],
        "LATE": ["mean"],
        "PAYMENT_RATIO": ["min", "max", "mean", "var"],
        "PAYMENT_DIFF": ["max", "mean", "sum", "var"],
        "AMT_INSTALMENT": ["max", "mean", "sum"],
        "AMT_PAYMENT": ["min", "max", "mean", "sum"],
        "DAYS_ENTRY_PAYMENT": ["max", "mean"],
    }
    out = aggregate(ins, "SK_ID_CURR", spec, "INSTAL")

    # Time window: the same behaviour restricted to the LAST YEAR.
    # Why: being late 8 years ago matters less than being late last month.
    # A lifetime mean dilutes recent deterioration; a window exposes it.
    recent = ins[ins["DAYS_INSTALMENT"] >= -365]
    recent_spec = {"DPD": ["max", "mean"], "LATE": ["mean"], "PAYMENT_RATIO": ["mean"]}
    out = out.join(aggregate(recent, "SK_ID_CURR", recent_spec, "INSTAL_1Y"))

    del ins, recent
    gc.collect()
    return out

In [ ]:
def credit_card_features():
    cc = reduce_mem(read("credit_card_balance"))

    # Utilisation = balance / limit. Near or above 1 → card maxed out,
    # a classic sign of financial stress.
    cc["UTILIZATION"] = safe_div(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"])

    cc, cc_cat = one_hot(cc)
    cc = cc.drop(columns="SK_ID_PREV")

    # Fewer hand-picked choices here: apply the same stats to every numeric
    # column and let the model sort out which ones matter.
    num_cols = [c for c in cc.columns if c not in cc_cat and c != "SK_ID_CURR"]
    spec = {c: ["min", "max", "mean", "sum", "var"] for c in num_cols}
    spec.update({c: ["mean"] for c in cc_cat})

    out = aggregate(cc, "SK_ID_CURR", spec, "CC")
    out["CC_N_MONTHS"] = cc.groupby("SK_ID_CURR").size()

    del cc
    gc.collect()
    return out

#TODO redundant features are present. fix them

In [ ]:
with timer("application"):
    df = application_features()

for name, fn in [("bureau", bureau_features),
                 ("previous_application", previous_features),
                 ("pos_cash", pos_cash_features),
                 ("installments", installments_features),
                 ("credit_card", credit_card_features)]:
    with timer(name):
        feats = reduce_mem(fn())
        # LEFT join: keep every application; no history → NaN ).
        df = df.join(feats, on="SK_ID_CURR", how="left")
        n_new = feats.shape[1]
        del feats
        gc.collect()
    print(f"   {name}: +{n_new} features → {df.shape}")

# Belt and braces: no inf should survive safe_div, but var/sum can overflow.
num_cols = df.select_dtypes("number").columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

# XGBoost rejects feature names containing characters like [ ] <, and one-hot
# names such as "..._Spouse, partner_MEAN" contain spaces and commas.
# Replace anything that isn't a letter, digit or underscore; keep names unique.
seen = {}
clean = []
for c in df.columns:
    new = re.sub(r"[^0-9A-Za-z_]+", "_", c)
    if new in seen:
        seen[new] += 1
        new = f"{new}_{seen[new]}"
    else:
        seen[new] = 0
    clean.append(new)
df.columns = clean

# Native categoricals: text → pandas "category". Because train and test are
# still stacked, each column gets ONE shared category list, so the integer
# code for "Pensioner" is identical in train and test.
cat_cols = [c for c in df.columns if is_text(df[c])]
for c in cat_cols:
    df[c] = df[c].astype("category")
print(f"{len(cat_cols)} categorical columns handled natively by XGBoost")

train_df = df[df["TARGET"].notna()].reset_index(drop=True)
test_df = df[df["TARGET"].isna()].reset_index(drop=True)
del df
gc.collect()

FEATURES = [c for c in train_df.columns if c not in ("TARGET", "SK_ID_CURR")]
print(f"train {train_df.shape} | test {test_df.shape} | {len(FEATURES)} features")

In [ ]:
PARAMS = dict(
    objective="binary:logistic",  # log-loss; g = p - y, h = p(1 - p)
    eval_metric="auc",            # what early stopping watches
    tree_method="hist",           # bucket each feature into max_bin bins, then
                                  # search splits over bins, not raw values
    max_bin=256,
    device=device,
    enable_categorical=True,      # split on "category" columns directly
    max_cat_to_onehot=4,          # <= 4 levels: split one level vs the rest;
                                  # more: sort levels by G/H of each level and
                                  # find the best partition of the sorted list

    learning_rate=0.02,           # each tree's leaf values are multiplied by
                                  # this. Small → many small corrections →
                                  # smoother fit, more trees needed
    n_estimators=10_000,          # upper bound; early stopping picks the real number
    early_stopping_rounds=200,    # stop if validation AUC hasn't improved in
                                  # 200 trees; keep the best iteration

    max_depth=6,                  # max interaction order a single tree can capture
    min_child_weight=30,          # min H per leaf
    gamma=0.0,                    # min gain to split (0 = rely on the others)
    reg_lambda=5.0,               # lambda in w* = -G/(H + lambda): shrinks leaf
                                  # values, most strongly in leaves with small H
    reg_alpha=0.05,               # L1 on leaf values: pushes tiny leaves to 0

    subsample=0.85,               # each tree sees 85% of rows    ┐ randomness →
    colsample_bytree=0.5,         # and 50% of columns             ┘ less correlated
                                  # trees. With ~800 highly redundant aggregate
                                  # features, column sampling also stops a few
                                  # EXT_SOURCE variants dominating every tree.
    random_state=SEED,
)


In [ ]:
train_df.to_parquet(os.path.join(OUT_DIR, 'train_features.parquet'))
test_df.to_parquet(os.path.join(OUT_DIR, 'test_features.parquet'))
print('cached →', OUT_DIR)


In [7]:
import wandb
wandb.login()

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: maharishabh7552 (maharishabh7552-dsv-global-transport-and-logistics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!pip install -q optuna optuna-integration

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_SEED  = 1234          # deliberately != SEED, so the tuning split differs from the CV folds
N_TRIALS   = 60
TUNE_LR    = 0.05          # search at a faster lr, train the final model at 0.02

# fixed across all trials — structural choices, not things to search
BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    max_bin=256,
    device=device,
    enable_categorical=True,
    early_stopping_rounds=100,
    n_estimators=4000,
)


In [ ]:
from sklearn.model_selection import train_test_split


tr_i, va_i = train_test_split(np.arange(len(X)), test_size=0.2,
                              stratify=y, random_state=TUNE_SEED)
X_tr, X_va = X.iloc[tr_i], X.iloc[va_i]
y_tr, y_va = y[tr_i], y[va_i]


def objective(trial):
    p = dict(
        BASE_PARAMS,
        learning_rate     = TUNE_LR,
        max_depth         = trial.suggest_int("max_depth", 4, 9),
        min_child_weight  = trial.suggest_float("min_child_weight", 5, 200, log=True),
        subsample         = trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree  = trial.suggest_float("colsample_bytree", 0.3, 0.9),
        colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        reg_lambda        = trial.suggest_float("reg_lambda", 1.0, 200.0, log=True),
        reg_alpha         = trial.suggest_float("reg_alpha", 1e-3, 20.0, log=True),
        gamma             = trial.suggest_float("gamma", 0.0, 5.0),
        max_cat_to_onehot = trial.suggest_int("max_cat_to_onehot", 2, 8),
    )
    m = xgb.XGBClassifier(**p)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    trial.set_user_attr("best_iteration", int(m.best_iteration))
    return m.best_score          # validation AUC at the best iteration


study = optuna.create_study(
    direction="maximize",
    study_name="xgb-home-credit",
    storage=f"sqlite:///{os.path.join(OUT_DIR, 'optuna.db')}",   # on Drive
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=TUNE_SEED, n_startup_trials=15),

)

study.optimize(objective, n_trials=N_TRIALS, timeout=45*60, show_progress_bar=True)

print(f"best AUC {study.best_value:.5f} over {len(study.trials)} trials")
print(f"baseline for reference: 0.79317 OOF (untuned)")
for k, v in study.best_params.items():
    print(f"  {k:18} {v}")


In [ ]:
# optuna.visualization.plot_param_importances(study).show()
# optuna.visualization.plot_parallel_coordinate(study).show()
# optuna.visualization.plot_optimization_history(study).show()


In [ ]:
# PARAMS = dict(
#     BASE_PARAMS,
#     **study.best_params,
#     learning_rate=0.02,           # back to the slow rate for the real run
#     n_estimators=10_000,
#     early_stopping_rounds=200,
# )
# print(PARAMS)
# TUNE_INFO = {
#     'tuned':             True,
#     'optuna_study':      study.study_name,
#     'optuna_trials':     len(study.trials),
#     'optuna_best_value': float(study.best_value),
#     'tune_seed':         TUNE_SEED,
#     'tune_lr':           TUNE_LR,
# }
# print(TUNE_INFO)


In [ ]:
from datetime import datetime, timezone
from wandb.integration.xgboost import WandbCallback

import wandb

X = train_df[FEATURES]
y = train_df['TARGET'].astype(int).to_numpy()
X_test = test_df[FEATURES]

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_gain = []
models = []
fold_scores = []

# GROUP = f"xgb-{'tuned' if TUNE_INFO['tuned'] else 'base'}-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
GROUP=f"xgb-'base'-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"


for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    with timer(f'fold {fold}'):
        run = wandb.init(
            project=WANDB_PROJECT,
            group=GROUP,
            name=f"fold{fold}",
            job_type="train",
            config={**PARAMS,
                    # **TUNE_INFO,
                    "fold": fold, "n_features": len(FEATURES),
                    "n_folds": N_FOLDS, "seed": SEED, "debug": DEBUG},
            reinit=True,
            # tags=['tuned' if TUNE_INFO['tuned'] else 'baseline'],
            tags='baseline',
        ) if log_wandb else None

        # fresh callback per fold — it carries state
        cbs = [WandbCallback(log_model=False, log_feature_importance=False)] if log_wandb else []
        model = xgb.XGBClassifier(**PARAMS, callbacks=cbs)

        model.fit(X.iloc[tr_idx], y[tr_idx],
                  eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=500)
        model.get_booster().set_param({"device": "cpu"})

        best = model.best_iteration
        rng = (0, best + 1)
        oof[va_idx] = model.predict_proba(X.iloc[va_idx], iteration_range=rng)[:, 1]
        test_pred += model.predict_proba(X_test, iteration_range=rng)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y[va_idx], oof[va_idx])
        print(f"fold {fold}: best_iteration {best} | AUC {fold_auc:.5f}")

        gain = model.get_booster().get_score(importance_type="gain")
        fold_gain.append(pd.Series(gain, name=f"fold{fold}"))

        models.append(model)
        fold_scores.append({'fold': fold, 'best_iteration': int(best),
                            'auc': float(fold_auc)})

        # survive a disconnect
        model.get_booster().save_model(os.path.join(OUT_DIR, f'fold{fold}.json'))
        np.save(os.path.join(OUT_DIR, 'oof_partial.npy'), oof)
        np.save(os.path.join(OUT_DIR, 'test_pred_partial.npy'), test_pred)

        if run:
            run.log({"fold_auc": fold_auc, "best_iteration": int(best)})
            run.finish()

        gc.collect()


In [ ]:
auc = roc_auc_score(y, oof)
pr_auc = average_precision_score(y, oof)
print(f"OOF ROC-AUC {auc:.5f}")
# PR-AUC (average precision) focuses on the positive class. Its no-skill
# baseline is the positive rate (~0.08), not 0.5 — compare against that.
print(f"OOF PR-AUC  {pr_auc:.5f}  (no-skill baseline {y.mean():.4f})")
# Calibration sanity check: because we didn't reweight classes, the mean
# predicted probability should be close to the true default rate.
print(f"mean predicted p {oof.mean():.4f} vs actual rate {y.mean():.4f}")

imp = pd.concat(fold_gain, axis=1).fillna(0)
imp["mean_gain"] = imp.mean(axis=1)
imp = imp.sort_values("mean_gain", ascending=False)
print(imp["mean_gain"].head(30))
imp.to_csv(os.path.join(OUT_DIR, "feature_importance_gain.csv"))

In [ ]:
pd.DataFrame({"SK_ID_CURR": train_df["SK_ID_CURR"], "TARGET": y, "OOF": oof}) \
    .to_csv(os.path.join(OUT_DIR, "oof_predictions.csv"), index=False)

# submission = pd.DataFrame({"SK_ID_CURR": test_df["SK_ID_CURR"].astype(int),
#                            "TARGET": test_pred})
# submission.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
# print(f"saved {len(submission)} test predictions")

In [8]:
!pip install -q shap
# !pip install -q wandb


In [ ]:

# The bundle carries everything needed to score a new applicant, not just the
# boosters: the exact FEATURES order, which columns are categorical and their
# training levels, the params, per-fold metrics and library versions.
#
# On categoricals: XGBoost 3.x matches category LABELS, so integer codes need
# not agree between training and inference (verified: the same rows predicted
# identically under three different category orderings). But the column must
# still BE categorical dtype — passing plain strings raises "The data type
# doesn't match the one used in the training dataset". Hence cat_levels.

import joblib
from datetime import datetime, timezone

cat_levels = {c: list(train_df[c].cat.categories)
            for c in FEATURES if str(train_df[c].dtype) == 'category'}

bundle = {
    'models':      models,            # the 5 fold models; prediction is their mean
    'features':    FEATURES,          # exact column order the models expect
    'cat_levels':  cat_levels,        # categorical columns + their training levels
    'params':      PARAMS,
    'n_folds':     N_FOLDS,
    'seed':        SEED,
    'fold_scores': fold_scores,
    'metrics': {'oof_roc_auc': float(auc), 'oof_pr_auc': float(pr_auc),
                'mean_pred': float(oof.mean()), 'positive_rate': float(y.mean())},
    'versions': {'xgboost': xgb.__version__, 'pandas': pd.__version__,
                'numpy': np.__version__, 'joblib': joblib.__version__},
    'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'debug': DEBUG,   # a 20k-row DEBUG run must never pass as a real model
    'wandb_group': GROUP if log_wandb else None,   # ties the bundle to its runs
    # 'tuning': TUNE_INFO,
}

bundle_path = os.path.join(OUT_DIR, 'xgb_home_credit.joblib')
joblib.dump(bundle, bundle_path, compress=3)
print(f'saved {bundle_path}  ({os.path.getsize(bundle_path)/1e6:.1f} MB)')

# A joblib pickle is tied to these library versions. Also write each booster in
# XGBoost's own JSON format, which loads in any 2.x/3.x build.
for i, m in enumerate(models):
    m.get_booster().save_model(os.path.join(OUT_DIR, f'booster_fold{i}.json'))


def predict_bundle(bundle, raw):
    """Score a raw DataFrame (categoricals as plain strings, e.g. straight from
    read_csv) with the shipped fold ensemble.
    A level unseen in training becomes NaN, which XGBoost routes as missing, so
    inference degrades instead of crashing; comparing incoming value sets
    against bundle['cat_levels'] is how you detect category drift."""
    X = raw.copy()
    for c, levels in bundle['cat_levels'].items():
        s = X[c].astype(object)
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    X = X[bundle['features']]
    return np.mean([m.predict_proba(X, iteration_range=(0, f['best_iteration'] + 1))[:, 1]
                    for m, f in zip(bundle['models'], bundle['fold_scores'])], axis=0)


# Self-check: reload from disk and confirm it reproduces this run's predictions.
check = joblib.load(bundle_path)
probe = test_df[FEATURES].head(2000).astype({c: 'object' for c in cat_levels})
diff = np.abs(predict_bundle(check, probe) - test_pred[:2000]).max()
print(f'reload check: max abs diff vs test_pred {diff:.2e}')
assert diff < 1e-6, 'shipped bundle does not reproduce this run'

# ---- Weights & Biases -----------------------------------------------------
# The five fold runs already logged their own curves under GROUP during
# training; this is the summary run that joins them and carries the artifact.
if log_wandb:
    run = wandb.init(project=WANDB_PROJECT, group=GROUP,
                    name='summary', job_type='eval',
                     tags=['tuned' if TUNE_INFO['tuned'] else 'baseline'],
                     notes='OOF AUC is optimistic twice over: early stopping picks the tree '
                      'count on the fold it scores, and hyperparameters were selected '
                      'on rows that overlap the CV folds.',
                    config={**PARAMS,**TUNE_INFO, 'n_features': len(FEATURES),
                            'n_folds': N_FOLDS, 'seed': SEED,
                            'n_train': len(train_df), 'n_test': len(test_df)})

    run.summary.update(bundle['metrics'])
    run.summary['fold_auc_std'] = float(np.std([f['auc'] for f in fold_scores]))

    top = imp['mean_gain'].head(50).rename_axis('feature').reset_index()
    run.log({'feature_importance': wandb.Table(dataframe=top)})

    # subsample for the ROC plot — 307k points makes a slow, unreadable chart
    sub = np.random.default_rng(SEED).choice(len(oof), size=20_000, replace=False)
    run.log({'roc': wandb.plot.roc_curve(y[sub], np.column_stack([1 - oof[sub], oof[sub]]),
                                        labels=['no default', 'default'])})

    art = wandb.Artifact('xgb-home-credit', type='model', metadata=bundle['metrics'])
    art.add_file(bundle_path)
    art.add_file(os.path.join(OUT_DIR, 'feature_importance_gain.csv'))
    # art.add_file(os.path.join(OUT_DIR, 'submission.csv'))
    run.log_artifact(art)
    run.finish()
    print(f'W&B run finished: {run.url}')


In [ ]:
import shap
sample = X.sample(5000, random_state=SEED)
dm = xgb.DMatrix(sample, enable_categorical=True)
sv = models[0].get_booster().predict(dm, pred_contribs=True)

base   = sv[0, -1]     # last column = base value
values = sv[0, :-1]    # everything else = per-feature contributions

shap.plots.waterfall(shap.Explanation(
    values=values,
    base_values=base,
    data=sample.iloc[0].values,
    feature_names=list(sample.columns),
))

In [ ]:
raw = models[0].get_booster().predict(dm, output_margin=True)
np.abs(sv.sum(axis=1) - raw).max()      # should be ~1e-5

In [9]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.4 MB/s eta 0:00:00


In [10]:


import lightgbm as lgb, catboost, wandb
from datetime import datetime, timezone
from scipy.stats import rankdata
print('lightgbm', lgb.__version__, '| catboost', catboost.__version__)

# X / y / X_test must be EXACTLY what the XGB loop used. Rebuild from the
# parquet if the runtime restarted — it round-trips the category dtypes.
if 'X' not in globals():
    train_df = pd.read_parquet(os.path.join(OUT_DIR, 'train_features.parquet'))
    test_df  = pd.read_parquet(os.path.join(OUT_DIR, 'test_features.parquet'))
    FEATURES = [c for c in train_df.columns if c not in ('TARGET', 'SK_ID_CURR')]
    X, X_test = train_df[FEATURES], test_df[FEATURES]
    y = train_df['TARGET'].astype(int).to_numpy()

# One folder of OOF vectors, one file per model, all row-aligned to X.
PRED_DIR = os.path.join(OUT_DIR, 'preds'); os.makedirs(PRED_DIR, exist_ok=True)

def save_preds(tag, oof_v, test_v):
    np.save(os.path.join(PRED_DIR, f'oof_{tag}.npy'),  oof_v)
    np.save(os.path.join(PRED_DIR, f'test_{tag}.npy'), test_v)
    print(f'{tag}: OOF ROC-AUC {roc_auc_score(y, oof_v):.5f}  → {PRED_DIR}')

if 'oof' in globals() and oof.any():          # still live from cell 26
    save_preds('xgb', oof, test_pred)
else:                                          # restarted — recover from disk
    _o = pd.read_csv(os.path.join(OUT_DIR, 'oof_predictions.csv'))
    save_preds('xgb', _o['OOF'].to_numpy(),
               np.load(os.path.join(OUT_DIR, 'test_pred_partial.npy')))

# Materialise the splits ONCE and reuse the list for both models. Same SEED
# would give the same folds anyway, but holding the list makes it impossible
# for a stray re-seed to silently de-align the OOF vectors.
FOLDS = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                             random_state=SEED).split(X, y))


lightgbm 4.6.0 | catboost 1.2.10
xgb: OOF ROC-AUC 0.79451  → /content/drive/MyDrive/HomeCredit_XGBoost/outputs/preds


In [11]:
from catboost import CatBoostClassifier

CAT_COLS = [c for c in FEATURES if str(X[c].dtype) == 'category']

# CatBoost won't take a pandas `category` column, and it rejects NaN inside a
# categorical feature outright. Integer codes it accepts directly — and because
# train and test were still stacked when the dtype was built, they share one
# category list, so the code for "Pensioner" is identical on both sides.
# NaN becomes -1, which is just another level.
def cb_frame(frame):
    out = frame.copy()
    for c in CAT_COLS:
        out[c] = out[c].cat.codes.astype('int32')
    return out

X_cb, X_test_cb = cb_frame(X), cb_frame(X_test)
gc.collect()

CB_PARAMS = dict(
    loss_function='Logloss',
    eval_metric='AUC',
    learning_rate=0.02,
    iterations=10_000,
    depth=6,                   # oblivious trees: every node at a given depth
                               # splits on the SAME feature/threshold. Weaker
                               # per tree than xgb's, which is the point — it's
                               # a differently-biased model, not a better one.
    l2_leaf_reg=5.0,
    random_seed=SEED,
    bootstrap_type='Bernoulli',
    subsample=0.85,
    one_hot_max_size=4,        # mirrors xgb's max_cat_to_onehot=4; above this
                               # CatBoost uses ordered target statistics
    border_count=254,
    task_type='GPU' if device == 'cuda' else 'CPU',
    verbose=500,
    allow_writing_files=False, # don't litter Drive with catboost_info/
)
if device != 'cuda':
    CB_PARAMS['rsm'] = 0.5     # column sampling is CPU-only; GPU+Logloss rejects it

oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))
cat_scores = []
GROUP_CAT = f"cat-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"

for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    with timer(f'cat fold {fold}'):
        run = wandb.init(
            project=WANDB_PROJECT, group=GROUP_CAT, name=f'fold{fold}',
            job_type='train', reinit=True, tags=['catboost'],
            config={**CB_PARAMS, 'fold': fold, 'n_features': len(FEATURES),
                    'n_folds': N_FOLDS, 'seed': SEED},
        ) if log_wandb else None

        model = CatBoostClassifier(**CB_PARAMS)
        model.fit(X_cb.iloc[tr_idx], y[tr_idx],
                  eval_set=(X_cb.iloc[va_idx], y[va_idx]),
                  cat_features=CAT_COLS,
                  early_stopping_rounds=200,
                  use_best_model=True)

        # use_best_model already truncated the ensemble at best_iteration,
        # so there's no iteration_range to pass here.
        oof_cat[va_idx] = model.predict_proba(X_cb.iloc[va_idx])[:, 1]
        test_cat += model.predict_proba(X_test_cb)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y[va_idx], oof_cat[va_idx])
        best = model.get_best_iteration()
        print(f'cat fold {fold}: best_iteration {best} | AUC {fold_auc:.5f}')
        cat_scores.append({'fold': fold, 'best_iteration': int(best),
                           'auc': float(fold_auc)})

        model.save_model(os.path.join(OUT_DIR, f'cat_fold{fold}.cbm'))
        np.save(os.path.join(PRED_DIR, 'oof_cat_partial.npy'), oof_cat)
        if run:
            run.log({'fold_auc': fold_auc, 'best_iteration': int(best)})
            run.finish()
        del model; gc.collect()

save_preds('cat', oof_cat, test_cat)
del X_cb, X_test_cb; gc.collect()          # ~1 GB back


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6939088	best: 0.6939088 (0)	total: 39.4ms	remaining: 6m 33s
500:	test: 0.7743412	best: 0.7743412 (500)	total: 15.8s	remaining: 4m 59s
1000:	test: 0.7810504	best: 0.7810504 (1000)	total: 31.1s	remaining: 4m 39s
1500:	test: 0.7836965	best: 0.7836965 (1500)	total: 46.2s	remaining: 4m 21s
2000:	test: 0.7855409	best: 0.7855409 (2000)	total: 1m 1s	remaining: 4m 4s
2500:	test: 0.7866473	best: 0.7866473 (2500)	total: 1m 16s	remaining: 3m 47s
3000:	test: 0.7873628	best: 0.7873683 (2997)	total: 1m 31s	remaining: 3m 32s
3500:	test: 0.7879229	best: 0.7879257 (3499)	total: 1m 46s	remaining: 3m 17s
4000:	test: 0.7885723	best: 0.7885728 (3999)	total: 2m 1s	remaining: 3m 2s
4500:	test: 0.7890566	best: 0.7890584 (4499)	total: 2m 16s	remaining: 2m 47s
5000:	test: 0.7891450	best: 0.7891641 (4986)	total: 2m 31s	remaining: 2m 31s
5500:	test: 0.7894021	best: 0.7894065 (5499)	total: 2m 47s	remaining: 2m 16s
6000:	test: 0.7895840	best: 0.7895979 (5982)	total: 3m 2s	remaining: 2m 1s
6500:	test: 0.78

best_iteration,▁
fold_auc,▁
best_iteration,9025
fold_auc,0.79063


[cat fold 0] 293s


...

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6631401	best: 0.6631401 (0)	total: 30.4ms	remaining: 5m 3s
500:	test: 0.7745522	best: 0.7745522 (500)	total: 16.1s	remaining: 5m 5s
1000:	test: 0.7817686	best: 0.7817686 (1000)	total: 31.5s	remaining: 4m 42s
1500:	test: 0.7844088	best: 0.7844100 (1494)	total: 46.6s	remaining: 4m 24s
2000:	test: 0.7860202	best: 0.7860250 (1997)	total: 1m 1s	remaining: 4m 7s
2500:	test: 0.7871467	best: 0.7871467 (2500)	total: 1m 16s	remaining: 3m 50s
3000:	test: 0.7878498	best: 0.7878763 (2976)	total: 1m 32s	remaining: 3m 35s
3500:	test: 0.7883788	best: 0.7883788 (3500)	total: 1m 47s	remaining: 3m 19s
4000:	test: 0.7889113	best: 0.7889146 (3996)	total: 2m 3s	remaining: 3m 4s
4500:	test: 0.7892242	best: 0.7892279 (4499)	total: 2m 18s	remaining: 2m 49s
5000:	test: 0.7895137	best: 0.7895572 (4963)	total: 2m 33s	remaining: 2m 33s
5500:	test: 0.7897007	best: 0.7897012 (5491)	total: 2m 49s	remaining: 2m 18s
6000:	test: 0.7898439	best: 0.7898847 (5965)	total: 3m 4s	remaining: 2m 2s
6500:	test: 0.7899

best_iteration,▁
fold_auc,▁
best_iteration,6430
fold_auc,0.79002


[cat fold 4] 213s
cat: OOF ROC-AUC 0.79275  → /content/drive/MyDrive/HomeCredit_XGBoost/outputs/preds


0

In [12]:
from wandb.integration.lightgbm import wandb_callback

LGB_PARAMS = dict(
    objective='binary',
    metric='auc',
    learning_rate=0.02,
    n_estimators=10_000,
    num_leaves=40,             # leaf-wise growth, so THIS is the capacity knob,
                               # not depth. xgb's depth-6 tree tops out at 64
                               # leaves; 40 keeps it a touch tighter.
    max_depth=-1,
    min_child_samples=50,
    min_child_weight=30,       # sum of hessians per leaf — same meaning as xgb
    reg_lambda=5.0,
    reg_alpha=0.05,
    subsample=0.85,
    subsample_freq=1,          # WITHOUT this, LightGBM silently ignores
                               # subsample and every tree sees all rows.
    colsample_bytree=0.5,
    max_bin=255,
    min_data_per_group=100,    # regularises the categorical splits: LightGBM's
    cat_smooth=20,             # cat handling overfits rare levels otherwise
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
lgb_scores = []
GROUP_LGB = f"lgb-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"

for fold, (tr_idx, va_idx) in enumerate(FOLDS):
    with timer(f'lgb fold {fold}'):
        run = wandb.init(
            project=WANDB_PROJECT, group=GROUP_LGB, name=f'fold{fold}',
            job_type='train', reinit=True, tags=['lightgbm'],
            config={**LGB_PARAMS, 'fold': fold, 'n_features': len(FEATURES),
                    'n_folds': N_FOLDS, 'seed': SEED},
        ) if log_wandb else None

        # LightGBM picks up the pandas `category` dtype by itself — no
        # categorical_feature argument needed, and no NaN handling either.
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        cbs = [lgb.early_stopping(200, verbose=False), lgb.log_evaluation(500)]
        if log_wandb:
            cbs.append(wandb_callback())
        model.fit(X.iloc[tr_idx], y[tr_idx],
                  eval_set=[(X.iloc[va_idx], y[va_idx])],
                  eval_metric='auc', callbacks=cbs)

        best = model.best_iteration_
        oof_lgb[va_idx] = model.predict_proba(X.iloc[va_idx], num_iteration=best)[:, 1]
        test_lgb += model.predict_proba(X_test, num_iteration=best)[:, 1] / N_FOLDS

        fold_auc = roc_auc_score(y[va_idx], oof_lgb[va_idx])
        print(f'lgb fold {fold}: best_iteration {best} | AUC {fold_auc:.5f}')
        lgb_scores.append({'fold': fold, 'best_iteration': int(best),
                           'auc': float(fold_auc)})

        model.booster_.save_model(os.path.join(OUT_DIR, f'lgb_fold{fold}.txt'),
                                  num_iteration=best)
        np.save(os.path.join(PRED_DIR, 'oof_lgb_partial.npy'), oof_lgb)
        if run:
            run.log({'fold_auc': fold_auc, 'best_iteration': int(best)})
            run.finish()
        del model; gc.collect()

save_preds('lgb', oof_lgb, test_lgb)


[500]	valid_0's auc: 0.786989
[1000]	valid_0's auc: 0.790399
[1500]	valid_0's auc: 0.791091
lgb fold 0: best_iteration 1600 | AUC 0.79125


best_iteration,▁
fold_auc,▁
iteration,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
valid_0_auc,▁▁▄▄▅▆▆▆████████████████████████████████
best_iteration,1600
fold_auc,0.79125
iteration,1799


[lgb fold 0] 206s


[500]	valid_0's auc: 0.796114
[1000]	valid_0's auc: 0.799742
[1500]	valid_0's auc: 0.800282
lgb fold 1: best_iteration 1410 | AUC 0.80034


best_iteration,▁
fold_auc,▁
iteration,▁▁▁▁▂▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▇▇▇▇▇█████
valid_0_auc,▁▁▄▄▅▆▆▆▇▇▇█████████████████████████████
best_iteration,1410
fold_auc,0.80034
iteration,1609


...

best_iteration,▁
fold_auc,▁
iteration,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇█████
valid_0_auc,▁▄▄▅▆▆▆▆▆▇▇▇▇▇██████████████████████████
best_iteration,1580
fold_auc,0.79549
iteration,1779


[lgb fold 3] 309s


[500]	valid_0's auc: 0.785725
[1000]	valid_0's auc: 0.790097
[1500]	valid_0's auc: 0.790932
lgb fold 4: best_iteration 1586 | AUC 0.79107


best_iteration,▁
fold_auc,▁
iteration,▁▁▁▁▂▂▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇███
valid_0_auc,▁▂▄▅▅▆▆▇▇▇▇▇████████████████████████████
best_iteration,1586
fold_auc,0.79107
iteration,1785


[lgb fold 4] 194s
lgb: OOF ROC-AUC 0.79389  → /content/drive/MyDrive/HomeCredit_XGBoost/outputs/preds


In [13]:
TAGS = ('xgb', 'lgb', 'cat')
oof_all  = {t: np.load(os.path.join(PRED_DIR, f'oof_{t}.npy'))  for t in TAGS}
test_all = {t: np.load(os.path.join(PRED_DIR, f'test_{t}.npy')) for t in TAGS}

for t in TAGS:
    print(f'{t:4s} OOF ROC-AUC {roc_auc_score(y, oof_all[t]):.5f}'
    f' AP {average_precision_score(y, oof_all[t]):.5f}')

# How much the blend can possibly gain is set by how much the models DISAGREE.
# Spearman on the OOF ranks is the number to look at: ~0.99 means they're the
# same model wearing hats and the blend will move the 4th decimal; <0.97 means
# there's real diversity to harvest.
print(pd.DataFrame(oof_all).corr(method='spearman').round(4))

# Rank-average, not probability-average: AUC only cares about ordering, and the
# three models are calibrated differently (CatBoost's ordered boosting in
# particular shifts the scale), so averaging raw probabilities lets whichever
# model is most confident dominate for no good reason.
def rank_norm(v):
    return rankdata(v) / len(v)

R_oof  = {t: rank_norm(oof_all[t])  for t in TAGS}
R_test = {t: rank_norm(test_all[t]) for t in TAGS}

blend_oof = sum(R_oof.values()) / 3
print(f'\nequal-weight rank blend OOF {roc_auc_score(y, blend_oof):.5f}')
# A rank-blend is tuned for ordering, i.e. for AUC. Check it didn't buy that
# by degrading the top of the ranking, which is the part AP actually measures
# and the part any real approve/decline threshold lives in.
print(f'blend AUC {roc_auc_score(y, blend_oof):.5f} '
      f' AP {average_precision_score(y, blend_oof):.5f} '
      f' (no-skill AP {y.mean():.4f})')

# Weight search on a 0.05 simplex grid. NOTE these weights are fitted on the
# same OOF they're scored on, so the number below is optimistic by roughly the
# amount it beats the equal-weight blend. If the gain over equal weights is
# under ~0.0005, ship equal weights — it won't generalise.
best = max(
    ((roc_auc_score(y, wx*R_oof['xgb'] + wl*R_oof['lgb'] + (1-wx-wl)*R_oof['cat']),
      wx, wl, round(1-wx-wl, 2))
     for wx in np.arange(0, 1.001, 0.05)
     for wl in np.arange(0, 1.001-wx+1e-9, 0.05)),
    key=lambda r: r[0])
print(f'best weights  xgb {best[1]:.2f}  lgb {best[2]:.2f}  cat {best[3]:.2f}'
      f'  → OOF {best[0]:.5f}  (equal-weight {roc_auc_score(y, blend_oof):.5f})')

W = {'xgb': 1/3, 'lgb': 1/3, 'cat': 1/3}          # ← swap in best[1:] if it earns it
blend_test = sum(W[t] * R_test[t] for t in TAGS)

pd.DataFrame({'SK_ID_CURR': train_df['SK_ID_CURR'], 'TARGET': y,
              **{f'oof_{t}': oof_all[t] for t in TAGS},
              'oof_blend': blend_oof}) \
  .to_csv(os.path.join(OUT_DIR, 'oof_all_models.csv'), index=False)

pd.DataFrame({'SK_ID_CURR': test_df['SK_ID_CURR'].astype(int),
              'TARGET': blend_test}) \
  .to_csv(os.path.join(OUT_DIR, 'submission_blend.csv'), index=False)
print('wrote oof_all_models.csv + submission_blend.csv →', OUT_DIR)


xgb  OOF ROC-AUC 0.79451 AP 0.29209
lgb  OOF ROC-AUC 0.79389 AP 0.28992
cat  OOF ROC-AUC 0.79275 AP 0.28871
        xgb     lgb     cat
xgb  1.0000  0.9784  0.9738
lgb  0.9784  1.0000  0.9665
cat  0.9738  0.9665  1.0000

equal-weight rank blend OOF 0.79573
blend AUC 0.79573  AP 0.29382  (no-skill AP 0.0807)
best weights  xgb 0.40  lgb 0.35  cat 0.25  → OOF 0.79576  (equal-weight 0.79573)
wrote oof_all_models.csv + submission_blend.csv → /content/drive/MyDrive/HomeCredit_XGBoost/outputs
